In [1]:
import os
import cv2
import torch
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import albumentations as A
from tqdm import tqdm

# =========================
# FIXED SEED
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# =========================
# FREE GPU MEMORY
# =========================
torch.cuda.empty_cache()

# =========================
# CONFIG - STAGE 2 (FINE-TUNING)
# =========================
IMAGE_DIR = "train/images"
LABEL_DIR = "train/labels"
VAL_IMAGE_DIR = "valid/images"
VAL_LABEL_DIR = "valid/labels"

class_names = ["helmet", "head", "body", "vest"]
NUM_CLASSES = len(class_names) + 1  # +1 for background

# =========================
# Stage 2 settings - reduced Learning Rate
# =========================
BATCH_SIZE = 2  # Fixed
EPOCHS = 12  # Only 10 additional epochs

# Reduce Learning Rate for Fine-tuning (5 times lower)
BACKBONE_LAYER4_LR = 0.0001  # From 0.0005 to 0.0001
RPN_HEAD_LR = 0.0005         # From 0.001 to 0.0005
ROI_HEAD_LR = 0.0005         # From 0.001 to 0.0005

WEIGHT_DECAY = 0.0005
MOMENTUM = 0.9

# Scheduler settings
LR_FACTOR = 0.5
LR_PATIENCE = 3
MIN_LR = 1e-7

# Gradient Clipping
GRAD_CLIP_MAX_NORM = 1.0

# =========================
# EARLY STOPPING - more patience (for Stage 2)
# =========================
EARLY_STOPPING_PATIENCE = 5  # From 4 to 5 for more patience

SAVE_DIR = "saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# Load the best model from Stage 1
# =========================
LOAD_PRETRAINED = True
PRETRAINED_MODEL_PATH = "saved_models/faster_rcnn_best.pth"  # Best model from Stage 1

# =========================
# AUGMENTATION (less for Stage 2)
# =========================
transform = A.Compose(
    [
        A.Resize(height=640, width=640),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.1),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.2),
        A.ShiftScaleRotate(shift_limit=0.03, scale_limit=0.05, rotate_limit=5, 
                          border_mode=cv2.BORDER_CONSTANT, p=0.3),
    ],
    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["labels"],
        min_visibility=0.3,
        check_each_transform=True,
        min_area=25,
    )
)

val_transform = A.Compose(
    [
        A.Resize(height=640, width=640),
    ],
    bbox_params=A.BboxParams(format="pascal_voc", label_fields=["labels"])
)

# =========================
# DATASET
# =========================
class CustomDataset(Dataset):
    def __init__(self, image_dir, label_dir, image_list, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_list = image_list
        self.transform = transform

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_name = self.image_list[idx]
        img_path = os.path.join(self.image_dir, img_name)
        label_path = os.path.join(self.label_dir, img_name.replace(".jpg", ".txt"))

        image = cv2.imread(img_path)
        if image is None:
            image = np.zeros((640, 640, 3), dtype=np.uint8)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        height, width = image.shape[:2]

        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls = int(parts[0])
                    xmin = float(parts[1])
                    ymin = float(parts[2])
                    xmax = float(parts[3])
                    ymax = float(parts[4])

                    xmin = max(0, min(width, xmin))
                    ymin = max(0, min(height, ymin))
                    xmax = max(0, min(width, xmax))
                    ymax = max(0, min(height, ymax))

                    if xmax <= xmin or ymax <= ymin:
                        continue

                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(cls + 1)

        if self.transform:
            if len(boxes) > 0:
                augmented = self.transform(image=image, bboxes=boxes, labels=labels)
                image = augmented["image"]
                boxes = augmented["bboxes"]
                labels = augmented["labels"]

                valid_boxes = []
                valid_labels = []
                for box, label in zip(boxes, labels):
                    if len(box) == 4:
                        xmin, ymin, xmax, ymax = box
                        if (xmax > xmin + 1 and ymax > ymin + 1 and 
                            xmin >= 0 and ymin >= 0 and 
                            xmax <= image.shape[1] and ymax <= image.shape[0]):
                            valid_boxes.append(box)
                            valid_labels.append(label)
                boxes = valid_boxes
                labels = valid_labels
            else:
                augmented = self.transform(image=image, bboxes=[], labels=[])
                image = augmented["image"]

        image = torch.from_numpy(image).float().permute(2, 0, 1) / 255.0

        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std

        if len(boxes) == 0:
            target = {
                "boxes": torch.zeros((0, 4), dtype=torch.float32),
                "labels": torch.zeros((0,), dtype=torch.int64)
            }
        else:
            target = {
                "boxes": torch.tensor(boxes, dtype=torch.float32),
                "labels": torch.tensor(labels, dtype=torch.int64)
            }

        return image, target

# =========================
# DATA LOADING
# =========================
def filter_images_with_boxes(image_dir, label_dir, image_list):
    valid_images = []
    total_boxes = 0
    
    for img in image_list:
        label_path = os.path.join(label_dir, img.replace(".jpg", ".txt"))
        if os.path.exists(label_path):
            with open(label_path) as f:
                lines = f.readlines()
                if len(lines) > 0:
                    valid_images.append(img)
                    total_boxes += len(lines)
    
    print(f"  Found {len(valid_images)} valid images with {total_boxes} total boxes")
    return valid_images

train_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith((".jpg", ".png", ".jpeg"))]
val_images = [f for f in os.listdir(VAL_IMAGE_DIR) if f.endswith((".jpg", ".png", ".jpeg"))]

print("Loading training data...")
train_images = filter_images_with_boxes(IMAGE_DIR, LABEL_DIR, train_images)

print("Loading validation data...")
val_images = filter_images_with_boxes(VAL_IMAGE_DIR, VAL_LABEL_DIR, val_images)

print(f"\nFinal counts:")
print(f"  Training samples: {len(train_images)}")
print(f"  Validation samples: {len(val_images)}")

def collate_fn(batch):
    return tuple(zip(*batch))

train_dataset = CustomDataset(IMAGE_DIR, LABEL_DIR, train_images, transform)
val_dataset = CustomDataset(VAL_IMAGE_DIR, VAL_LABEL_DIR, val_images, val_transform)

print("\nCreating DataLoaders...")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
    pin_memory=False
)

print("✅ DataLoaders created successfully!")

# =========================
# MODEL
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

# Clear cache before model loading
torch.cuda.empty_cache()

model = fasterrcnn_resnet50_fpn(weights=None)  # Without initial weights
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

# =========================
# Load the best model from Stage 1
# =========================
if LOAD_PRETRAINED and os.path.exists(PRETRAINED_MODEL_PATH):
    print("\n" + "="*70)
    print("LOADING BEST MODEL FROM STAGE 1")
    print("="*70)
    try:
        state_dict = torch.load(PRETRAINED_MODEL_PATH, map_location=device)
        model.load_state_dict(state_dict, strict=False)
        print(f"  ✅ Successfully loaded model from: {PRETRAINED_MODEL_PATH}")
        print(f"  📊 This model achieved mAP: 0.2464 in Stage 1")
        print(f"  🚀 Continuing training with lower LR for fine-tuning")
    except Exception as e:
        print(f"  ❌ Failed to load model: {e}")
        print("  🔄 Starting from scratch instead")
        LOAD_PRETRAINED = False
else:
    print(f"\n⚠️ Model not found at: {PRETRAINED_MODEL_PATH}")
    print("  🔄 Starting from scratch")

model.to(device)

# =========================
# LAYER FREEZING - UNFREEZE MORE LAYERS
# =========================
print("\n" + "="*70)
print("LAYER FREEZING CONFIGURATION - STAGE 2")
print("="*70)

# Freeze all layers initially
for param in model.parameters():
    param.requires_grad = False

# =========================
# UNFREEZE layer3 and layer4 (more than Stage 1)
# =========================
print("\n📌 Backbone - layer3 and layer4 fine-tuned:")
layer_count = 0
for name, param in model.backbone.body.named_parameters():
    if 'layer3' in name or 'layer4' in name:
        param.requires_grad = True
        layer_count += 1
print(f"  Unfrozen: {layer_count} parameters")

# Unfreeze RPN
print("\n📌 RPN - ALL layers fine-tuned:")
rpn_count = 0
for param in model.rpn.parameters():
    param.requires_grad = True
    rpn_count += 1
print(f"  Unfrozen: {rpn_count} parameters")

# Unfreeze ROI Heads
print("\n📌 ROI Heads - ALL layers fine-tuned:")
roi_count = 0
for param in model.roi_heads.parameters():
    param.requires_grad = True
    roi_count += 1
print(f"  Unfrozen: {roi_count} parameters")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "-"*70)
print(f"📊 Total parameters: {total_params:,}")
print(f"📊 Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
print("="*70 + "\n")

# =========================
# OPTIMIZER - WITH LOWER LR
# =========================
optimizer_params = [
    {
        'params': [p for n, p in model.backbone.body.named_parameters() 
                  if ('layer3' in n or 'layer4' in n) and p.requires_grad],
        'lr': BACKBONE_LAYER4_LR,  # 0.0001
    },
    {
        'params': model.rpn.parameters(),
        'lr': RPN_HEAD_LR,  # 0.0005
    },
    {
        'params': model.roi_heads.parameters(),
        'lr': ROI_HEAD_LR,  # 0.0005
    },
]

optimizer = torch.optim.SGD(optimizer_params, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE,
    threshold=1e-3, cooldown=2, min_lr=MIN_LR, verbose=True
)

metric = MeanAveragePrecision()

# =========================
# EARLY STOPPING
# =========================
class EarlyStopping:
    def __init__(self, patience=5, mode='max', min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = -float('inf') if mode == 'max' else float('inf')
        self.mode = mode
        self.early_stop = False

    def step(self, score):
        if self.mode == 'max':
            is_better = score > self.best_score + self.min_delta
        else:
            is_better = score < self.best_score - self.min_delta

        if is_better:
            self.best_score = score
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                return True
        return False

    def get_status(self):
        return {
            'counter': self.counter,
            'patience': self.patience,
            'best_score': self.best_score,
            'remaining': self.patience - self.counter
        }

early_stopper = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, mode='max', min_delta=0.001)

# =========================
# TRAINING LOOP - STAGE 2
# =========================
print("\n" + "="*70)
print("🚀 STARTING STAGE 2 TRAINING")
print("="*70)
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rates (Reduced for fine-tuning):")
print(f"  - Backbone (layer3+4): {BACKBONE_LAYER4_LR} (was 0.0005)")
print(f"  - RPN: {RPN_HEAD_LR} (was 0.001)")
print(f"  - ROI Heads: {ROI_HEAD_LR} (was 0.001)")
print(f"Early Stopping Patience: {EARLY_STOPPING_PATIENCE} epochs")
print(f"Starting from best model with mAP: 0.2464")
print("="*70 + "\n")

best_map = 0
accumulation_steps = 2

for epoch in range(EPOCHS):
    print(f"\n{'='*70}")
    print(f"🚀 STAGE 2 - EPOCH {epoch+1}/{EPOCHS}")
    print(f"{'='*70}")
    
    # -----------------
    # TRAIN PHASE
    # -----------------
    model.train()
    train_loss = 0
    train_losses = {}
    train_num_batches = 0
    
    torch.cuda.empty_cache()

    pbar = tqdm(train_loader, desc=f"Training", unit='batch', 
                ncols=100, position=0, leave=True)
    
    optimizer.zero_grad()
    
    for batch_idx, (images, targets) in enumerate(pbar):
        try:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())
            
            loss = loss / accumulation_steps
            loss.backward()

            train_loss += loss.item() * accumulation_steps
            for k, v in loss_dict.items():
                if k not in train_losses:
                    train_losses[k] = 0
                train_losses[k] += v.item()
            train_num_batches += 1

            if (batch_idx + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_MAX_NORM)
                optimizer.step()
                optimizer.zero_grad()

            pbar.set_postfix({
                'loss': f'{loss.item() * accumulation_steps:.4f}',
                'lr': f'{optimizer.param_groups[0]["lr"]:.6f}'
            })

            if torch.isnan(loss):
                print(f"\n❌ NaN detected at batch {batch_idx+1}!")
                raise RuntimeError("NaN detected!")
                
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"\n⚠️ CUDA OOM at batch {batch_idx}. Clearing cache...")
                torch.cuda.empty_cache()
                continue
            else:
                print(f"\n❌ Error in batch {batch_idx}: {e}")
                continue

    if (batch_idx + 1) % accumulation_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_MAX_NORM)
        optimizer.step()
        optimizer.zero_grad()

    pbar.close()

    train_loss_avg = train_loss / train_num_batches if train_num_batches > 0 else 0
    train_losses_avg = {k: v / train_num_batches for k, v in train_losses.items()} if train_num_batches > 0 else {}

    # -----------------
    # VALIDATION PHASE
    # -----------------
    print(f"\n🔄 Validating...")
    model.eval()
    metric.reset()

    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc="Validation", unit='batch', ncols=100, position=0, leave=True)
        for images, targets in val_pbar:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            preds = model(images)
            metric.update(preds, targets)
            
            val_pbar.set_postfix({'status': 'processing'})
        
        val_pbar.close()

    results = metric.compute()
    current_map = results['map'].item()
    map_50 = results['map_50'].item()
    map_75 = results.get('map_75', torch.tensor(0.0)).item()

    scheduler.step(current_map)
    early_stop = early_stopper.step(current_map)
    status = early_stopper.get_status()

    # -----------------
    # PRINT RESULTS
    # -----------------
    print(f"\n📊 STAGE 2 - RESULTS EPOCH {epoch+1}:")
    print(f"  Training Loss: {train_loss_avg:.4f}")
    if train_losses_avg:
        for k, v in train_losses_avg.items():
            print(f"    - {k}: {v:.4f}")
    print(f"\n  Validation mAP:")
    print(f"    - mAP@0.5:0.95: {current_map:.4f}")
    print(f"    - mAP@0.5: {map_50:.4f}")
    print(f"    - mAP@0.75: {map_75:.4f}")
    print(f"\n  📊 Early Stopping:")
    print(f"    - Best mAP: {status['best_score']:.4f}")
    print(f"    - No improvement: {status['counter']}/{status['patience']}")
    print(f"    - Remaining: {status['remaining']}")

    # Save best model
    if current_map > best_map:
        best_map = current_map
        best_model_path = os.path.join(SAVE_DIR, "faster_rcnn_best_stage2.pth")
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✅ New best model saved! (mAP: {best_map:.4f})")

    if early_stop:
        print("\n" + "="*70)
        print("🛑 EARLY STOPPING TRIGGERED!")
        print(f"  No improvement for {EARLY_STOPPING_PATIENCE} epochs")
        print(f"  Best mAP: {status['best_score']:.4f}")
        print("="*70)
        break

# =========================
# SAVE FINAL MODEL - STAGE 2
# =========================
print("\n" + "="*70)
print("SAVING FINAL MODEL - STAGE 2")
print("="*70)

final_model_path = os.path.join(SAVE_DIR, "faster_rcnn_final_stage2.pth")
torch.save(model.state_dict(), final_model_path)

file_size = os.path.getsize(final_model_path) / (1024 * 1024)
print(f"  💾 Final model saved: {final_model_path}")
print(f"  📁 File size: {file_size:.2f} MB")

print("\n" + "="*70)
print("STAGE 2 TRAINING COMPLETE!")
print("="*70)

print(f"\n📈 STAGE 2 SUMMARY:")
print(f"  Total epochs: {epoch+1}")
print(f"  Best mAP: {best_map:.4f}")
print(f"  Final mAP: {current_map:.4f}")
print(f"  Improvement from Stage 1: {best_map - 0.2464:.4f}")

print(f"\n📁 Saved files:")
print(f"  - {final_model_path} ({file_size:.2f} MB)")
if best_map > 0:
    best_path = os.path.join(SAVE_DIR, "faster_rcnn_best_stage2.pth")
    best_size = os.path.getsize(best_path) / (1024 * 1024)
    print(f"  - {best_path} ({best_size:.2f} MB) - BEST MODEL")

print("\n🎯 STAGE 2 CONFIGURATION:")
print(f"  ✅ Learning Rates reduced by 5x")
print(f"  ✅ Unfroze layer3 AND layer4 (more than Stage 1)")
print(f"  ✅ Starting from best model (mAP: 0.2464)")
print(f"  ✅ Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")
print("="*70)

Loading training data...
  Found 1829 valid images with 4862 total boxes
Loading validation data...
  Found 457 valid images with 1176 total boxes

Final counts:
  Training samples: 1829
  Validation samples: 457

Creating DataLoaders...
✅ DataLoaders created successfully!

Using device: cuda

LOADING BEST MODEL FROM STAGE 1
  ✅ Successfully loaded model from: saved_models/faster_rcnn_best.pth
  📊 This model achieved mAP: 0.2464 in Stage 1
  🚀 Continuing training with lower LR for fine-tuning

LAYER FREEZING CONFIGURATION - STAGE 2

📌 Backbone - layer3 and layer4 fine-tuned:
  Unfrozen: 29 parameters

📌 RPN - ALL layers fine-tuned:
  Unfrozen: 6 parameters

📌 ROI Heads - ALL layers fine-tuned:
  Unfrozen: 8 parameters

----------------------------------------------------------------------
📊 Total parameters: 41,314,536
📊 Trainable parameters: 36,535,336 (88.43%)


🚀 STARTING STAGE 2 TRAINING
GPU: NVIDIA GeForce GTX 1660 Ti
Batch Size: 2
Learning Rates (Reduced for fine-tuning):
  - Bac

Training: 100%|██████████████████████| 915/915 [06:37<00:00,  2.30batch/s, loss=0.1928, lr=0.000100]



🔄 Validating...


Validation: 100%|███████████████████████████| 229/229 [00:55<00:00,  4.16batch/s, status=processing]



📊 STAGE 2 - RESULTS EPOCH 1:
  Training Loss: 0.3249
    - loss_classifier: 0.1475
    - loss_box_reg: 0.1444
    - loss_objectness: 0.0242
    - loss_rpn_box_reg: 0.0088

  Validation mAP:
    - mAP@0.5:0.95: 0.2574
    - mAP@0.5: 0.4584
    - mAP@0.75: 0.2647

  📊 Early Stopping:
    - Best mAP: 0.2574
    - No improvement: 0/5
    - Remaining: 5
  ✅ New best model saved! (mAP: 0.2574)

🚀 STAGE 2 - EPOCH 2/12


...


📊 STAGE 2 - RESULTS EPOCH 11:
  Training Loss: 0.3151
    - loss_classifier: 0.1435
    - loss_box_reg: 0.1422
    - loss_objectness: 0.0213
    - loss_rpn_box_reg: 0.0081

  Validation mAP:
    - mAP@0.5:0.95: 0.2927
    - mAP@0.5: 0.5022
    - mAP@0.75: 0.3212

  📊 Early Stopping:
    - Best mAP: 0.2927
    - No improvement: 0/5
    - Remaining: 5
  ✅ New best model saved! (mAP: 0.2927)

🚀 STAGE 2 - EPOCH 12/12


Training: 100%|██████████████████████| 915/915 [06:31<00:00,  2.34batch/s, loss=0.9079, lr=0.000100]



🔄 Validating...


Validation: 100%|███████████████████████████| 229/229 [00:52<00:00,  4.32batch/s, status=processing]



📊 STAGE 2 - RESULTS EPOCH 12:
  Training Loss: 0.3114
    - loss_classifier: 0.1419
    - loss_box_reg: 0.1418
    - loss_objectness: 0.0198
    - loss_rpn_box_reg: 0.0079

  Validation mAP:
    - mAP@0.5:0.95: 0.2889
    - mAP@0.5: 0.5022
    - mAP@0.75: 0.3100

  📊 Early Stopping:
    - Best mAP: 0.2927
    - No improvement: 1/5
    - Remaining: 4

SAVING FINAL MODEL - STAGE 2
  💾 Final model saved: saved_models\faster_rcnn_final_stage2.pth
  📁 File size: 158.11 MB

STAGE 2 TRAINING COMPLETE!

📈 STAGE 2 SUMMARY:
  Total epochs: 12
  Best mAP: 0.2927
  Final mAP: 0.2889
  Improvement from Stage 1: 0.0463

📁 Saved files:
  - saved_models\faster_rcnn_final_stage2.pth (158.11 MB)
  - saved_models\faster_rcnn_best_stage2.pth (158.11 MB) - BEST MODEL

🎯 STAGE 2 CONFIGURATION:
  ✅ Learning Rates reduced by 5x
  ✅ Unfroze layer3 AND layer4 (more than Stage 1)
  ✅ Starting from best model (mAP: 0.2464)
  ✅ Early Stopping Patience: 5
